# 实验 1：随机梯度的均值、均方与批量缩放

**对应第3讲；课堂实验约1学时。** 本 Notebook 调用 `experiments/lab01_sgd.py`，不另写一套训练实现。默认先运行一个随机种子和较少的蒙特卡洛轨迹，再用精确递推检查结论。

要回答的主问题是：平均误差趋于零，是否足以说明随机训练稳定？本实验把三个对象分开：参数误差的均值、误差平方的期望，以及有限条轨迹的经验平均。

**设备说明：** 当前实验脚本使用 NumPy，在 CPU 上运行。环境报告即使显示 CUDA 可用，也不表示这段 NumPy 模拟运行在 GPU 上；后两个实验会自动选择可用的 CUDA 或 CPU。


## 环境与执行方式

在已有的 Jupyter 或 VS Code Notebook 环境中，选择安装了课程依赖的 Python 内核，按顺序运行单元格。计算依赖与命令行脚本相同；展示使用 Notebook 环境已有的 `IPython.display`，不需要 pandas 或额外的交互组件。

如尚未安装依赖，请在**课程目录的终端**手动运行：

```bash
python -m pip install -r requirements.txt
```

本 Notebook 不会自动安装包，也不会下载外部数据集。生成的图、CSV 和 JSON 写入课程目录下的 `results/notebooks/`，不会写入 Notebook 的源码单元格。仓库版本有意保留空输出；运行后可以通过生成文件复核结果。


## 课前：先写出预测，再看曲线

设 $x\sim\mathcal N(0,1)$、$y=a_\star x+\varepsilon$，每步独立取 $B$ 个新样本，批次损失为平均平方误差的一半。误差 $e_t=a_t-a_\star$ 的精确递推为

$$
\mathbb E[e_{t+1}]=(1-\eta)\mathbb E[e_t],\qquad
q_{t+1}=\left[(1-\eta)^2+\frac{2\eta^2}{B}\right]q_t+rac{\eta^2\tau^2}{B},
$$

其中 $q_t=\mathbb E[e_t^2]$、$\tau$ 是标签噪声的标准差。

运行前回答三问：

1. 取 $B=1,\eta=0.9,\tau=0$，均值的乘子与均方的乘子分别是多少？
2. 为什么均方稳定边界 $\eta<2B/(B+2)$ 与均值稳定边界 $\eta<2$ 不同？
3. 如果少量极端轨迹主导二阶矩，增加轨迹数是否保证每一次经验曲线都更接近理论曲线？

后面的结果用于检查这些预测。不要根据图形反过来挑选“应该出现”的答案。


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import json
import sys
from IPython.display import display, Image, JSON, Markdown

# 可从 notebooks/、课程目录或仓库根目录启动，不依赖机器上的绝对路径。
start = Path.cwd().resolve()
candidates = []
for parent in (start, *start.parents):
    candidates.extend((parent, parent / '2027' / 'nonlinear optimization'))
COURSE_ROOT = next(
    (p for p in candidates
     if (p / 'experiments' / 'lab01_sgd.py').is_file()
     and (p / 'chapters').is_dir()),
    None,
)
if COURSE_ROOT is None:
    raise FileNotFoundError('未找到课程目录：请从本仓库的课程目录或 notebooks 目录启动。')
EXPERIMENTS = COURSE_ROOT / 'experiments'
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))
print('课程目录：', COURSE_ROOT)
print('实验实现：', EXPERIMENTS)


In [ ]:
from lab01_sgd import Config as LabConfig, run as run_lab
from lab01_sgd import exact_moments, second_moment_factor

config = LabConfig(seed=2027, paths=5000, steps=20, noise_std=0.2)
OUTPUT = COURSE_ROOT / 'results' / 'notebooks' / 'lab01' / f'seed_{config.seed}'
display(JSON({'config': asdict(config), 'output': str(OUTPUT), 'actual_compute': 'NumPy / CPU'}))


## 课中 1：运行共同实现

默认 `paths=5000, steps=20` 用于快速演示；命令行脚本默认是更多轨迹和更长的观察窗口。

脚本的三幅图对应不同问题：左图固定 $B=1,\eta=0.9$，**不加标签噪声**，展示乘性噪声；中图用单步蒙特卡洛核对不同批量的均方乘子；右图使用 `noise_std` 指定的标签噪声，比较固定样本预算下的精确二阶矩。修改 `noise_std` 不会改变左图的无标签噪声设定。


In [ ]:
report = run_lab(config, output=OUTPUT)
display(Image(filename=str(OUTPUT / 'lab01_sgd.png')))


In [ ]:
saved_report = json.loads((OUTPUT / 'report.json').read_text(encoding='utf-8'))
display(JSON({
    'config': saved_report['config'],
    'environment': saved_report['environment'],
    'interpretation': saved_report['interpretation'],
}))
display(JSON(saved_report['one_step_checks']))


## 课中 2：把稳定边界算成数值

下面的表只使用精确公式，因此不受有限轨迹数影响。`second_moment_factor` 是一次更新中乘在旧二阶矩上的系数；只有它小于1，均方递推才收缩。边界处若标签噪声非零，二阶矩仍会线性增长。


In [ ]:
stability_rows = []
for batch in (1, 4, 16):
    for eta in (0.3, 0.65, 0.9, 1.5, 1.8):
        factor = float(second_moment_factor(eta, batch))
        stability_rows.append({
            'batch': batch,
            'eta': eta,
            'mean_multiplier': 1 - eta,
            'mean_stable': abs(1 - eta) < 1,
            'mean_square_factor': factor,
            'mean_square_contracts': factor < 1,
            'mean_square_eta_limit': 2 * batch / (batch + 2),
        })
display(JSON(stability_rows))


## 课中 3：修改一个设置，检查有限时间与稳态

可以修改下一格的 `B_USER`、`ETA_USER`、`TAU_USER` 和 `T_USER`。这格调用同一个精确递推函数，不重新运行完整模拟。

先预测：在同样的 $\eta/B$ 下，为什么精确稳态仍可能不同？答案应涉及

$$q_\infty=\frac{\eta\tau^2}{B[2-\eta(1+2/B)]},$$

而不能只保留分子中的比值。当均方不稳定时，下面不会给出一个没有意义的“稳态值”。


In [ ]:
B_USER = 4
ETA_USER = 0.2
TAU_USER = 0.2
T_USER = 100

if B_USER < 1 or T_USER < 1 or ETA_USER <= 0 or TAU_USER < 0:
    raise ValueError('批量、步数和学习率应为正，噪声标准差应非负。')
mean, second = exact_moments(ETA_USER, B_USER, T_USER, noise_std=TAU_USER)
factor = float(second_moment_factor(ETA_USER, B_USER))
stationary_second = (
    ETA_USER * TAU_USER**2 / (B_USER * (2 - ETA_USER * (1 + 2 / B_USER)))
    if factor < 1 else None
)
display(JSON({
    'batch': B_USER, 'eta': ETA_USER, 'noise_std': TAU_USER,
    'updates': T_USER, 'processed_samples': B_USER * T_USER,
    'mean_square_factor': factor,
    'finite_time_mean': float(mean[-1]),
    'finite_time_second_moment': float(second[-1]),
    'stationary_second_moment_if_stable': stationary_second,
    'finite_time_excess_risk': float(second[-1] / 2),
}))


## 课中 4：查看可复核的原始记录

`moment_paths.csv` 同时保留经验均值、经验二阶矩、绝对误差中位数和理论值。中位数有助于描述典型轨迹，但不能替代二阶矩来判断平均平方风险。


In [ ]:
import csv

with (OUTPUT / 'moment_paths.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
display(JSON({'first_rows': rows[:3], 'last_rows': rows[-3:]}))
print('批量缩放的逐步理论数据：', OUTPUT / 'batch_scaling.csv')


## 课后：报告问题与核验要点

提交一页报告，说明模型、抽样方式、步长、批量、轨迹数和计算预算，并回答：

1. $B=1,\eta=0.9$ 为什么出现均值收敛、均方发散？核验值为均值乘子 $0.1$、均方乘子 $1.63$。
2. 哪些经验曲线偏离理论期望？增加轨迹数、改变随机种子和缩短时间窗口分别能检查什么？不得删除大误差轨迹后仍声称计算的是原分布二阶矩。
3. 线性学习率缩放的三条精确曲线为何不必重合？应同时讨论稳定分母与固定样本预算。
4. 如何区分不可约标签噪声、有限步优化误差和随机迭代的稳态误差？

**可继续执行的检查：** 修改默认配置的 `paths` 或 `seed` 后重新运行，使用新的输出目录保留旧结果。对乘性不稳定过程，有限样本漏掉极端事件是可预期的现象，不能把一次经验均值解释为真实期望的精确值。

本实验没有优化器优劣排名；它训练的是用可解模型校验随机优化结论的能力。
